In [ ]:
!pip install zarr fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.3/405.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.2 MB/s eta 0:00:00


In [ ]:
!pip install metpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.4/424.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.9/351.9 kB 33.8 MB/s eta 0:00:00


In [ ]:
!pip install cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 72.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tarfile
import xarray as xr
import pandas as pd
import numpy as np
import xarray as xr
import glob
import zarr
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torchvision.models as models
import copy, json, random
import metpy.calc as mpcalc
from metpy.units import units
from scipy.stats import pearsonr
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader


For daily temporal res


In [ ]:
ROWS = slice(15, 95)
COLS = slice(110, 210)

def extract_day(path, date="2016-01-01", rows=None, cols=None,
                squeeze_lay=False, out=None):
    with xr.open_dataset(path) as ds:

        if ds.sizes["TSTEP"] == 1:
            # static file: drop the time dimension
            sub = ds.isel(TSTEP=0)
        else:
            # decode TFLAG (YYYYDDD, HHMMSS) into real datetimes
            tf = ds["TFLAG"].isel(VAR=0).values
            t = pd.to_datetime(tf[:, 0].astype(str), format="%Y%j") \
                + pd.to_timedelta(tf[:, 1] // 10000, unit="h")

            day = pd.Timestamp(date)
            keep = (t >= day) & (t < day + pd.Timedelta(days=1))
            sub = ds.isel(TSTEP=keep).assign_coords(TSTEP=t[keep])

        # optional spatial window (skip for the Texas subset files)
        if rows is not None or cols is not None:
            sub = sub.isel(ROW=rows or slice(None), COL=cols or slice(None))

        sub = sub.drop_vars("TFLAG")
        if squeeze_lay and "LAY" in sub.dims and sub.sizes["LAY"] == 1:
            sub = sub.squeeze("LAY", drop=True)

        sub = sub.load()          # read into memory before the file closes

    if out:
        sub.to_netcdf(out)
    return sub

For hourly temoral resolution

In [ ]:
def extract_day_texas(path, rows=ROWS, cols=COLS, squeeze_lay=True, out=None):
    with xr.open_dataset(path) as ds:

        # 1. Decode TFLAG into real datetime coordinates (keep ALL timesteps in the file)
        if "TFLAG" in ds.variables:
            tf = ds["TFLAG"].isel(VAR=0).values
            hh = tf[:, 1] // 10000
            t = pd.to_datetime(tf[:, 0].astype(str), format="%Y%j") + pd.to_timedelta(hh, unit="h")

            sub = ds.assign_coords(TSTEP=t)
            sub = sub.drop_vars("TFLAG")
        else:
            sub = ds

        # 2. Apply spatial window for Texas
        if rows is not None or cols is not None:
            sub = sub.isel(ROW=rows or slice(None), COL=cols or slice(None))

        # 3. Drop single layer if NLAYS = 1
        if squeeze_lay and "LAY" in sub.dims and sub.sizes["LAY"] == 1:
            sub = sub.squeeze("LAY", drop=True)

        # 4. Load into memory before file closes
        sub = sub.load()

    if out:
        sub.to_netcdf(out)


    return sub

for static


In [ ]:
def extract_static_file(path, rows=ROWS, cols=COLS, out=None):
    """Processes a static/multi-layer IOAPI file (like LUFRAC) for Texas."""
    with xr.open_dataset(path) as ds:

        # 1. Static file check: drop the time dimension since TSTEP = 1
        if "TSTEP" in ds.dims and ds.sizes["TSTEP"] == 1:
            sub = ds.isel(TSTEP=0)
        else:
            sub = ds

        # 2. Apply spatial window for Texas (ROWS and COLS)
        if rows is not None or cols is not None:
            sub = sub.isel(ROW=rows or slice(None), COL=cols or slice(None))

        # 3. Clean up TFLAG if present
        if "TFLAG" in sub.variables:
            sub = sub.drop_vars("TFLAG")

        # Note: We DO NOT squeeze layers here because NLAYS = 20 (multi-layer data)

        # 4. Load into memory before file closes
        sub = sub.load()

    if out:
        sub.to_netcdf(out)

    return sub

In [ ]:
file_id = "fileID"
url = f'https://drive.google.com/uc?id={file_id}'
output_path = 'label.nc4'

gdown.download(url, output_path, fuzzy=True)


In [ ]:
met  = extract_day("/content/label.nc4",   rows=ROWS, cols=COLS, squeeze_lay=True, out="label_texas016jan2.nc4")

In [ ]:
file_url = "/content/drive/MyDrive/Copy of MCIPv51_WRFv411_noltng_12US1.35L.20160115.tar"
# output_tar = "my_archive.tar"

# Download the file
# gdown.download(file_url, output_tar, quiet=False, fuzzy=True)

# 2. Inspect the contents without extracting
print("--- Files inside the archive ---")
with tarfile.open(file_url, 'r') as tar:
    # List all file names inside the tar archive
    for member_name in tar.getnames()[:50]:  # Limits preview to the first 50 files so it doesn't spam your screen
        print(member_name)

    total_files = len(tar.getnames())
    print(f"\nTotal files in archive: {total_files}")

In [ ]:
with tarfile.open(file_url, 'r') as tar:
    tar.extract('./METCRO2D_20160115.nc4')
    tar.extract('./LUFRAC_CRO_20160115.nc4')
print("Extracted files successfully!")

In [ ]:
met01  = extract_day_texas("/content/METCRO2D_20160115.nc4", rows=ROWS, cols=COLS,squeeze_lay=True, out="met_texas016jan15.nc4")
met02  = extract_static_file("/content/LUFRAC_CRO_20160115.nc4", rows=ROWS, cols=COLS, out="lu_texas016jan15.nc4")

In [ ]:
spatial_file = "/content/grid_texas016jan1.nc4"  # Replace with your actual file name
output_file = "texas_spatial_features_clean.nc4"

with xr.open_dataset(spatial_file) as ds:

    # 2. Drop the unwanted variables
    vars_to_drop = ["LWMASK", "MSFX2"]
    ds_clean = ds.drop_vars(vars_to_drop, errors="ignore")

    # 3. Save the cleaned dataset
    ds_clean.to_netcdf(output_file)

For emission

In [ ]:
TARGET_VARS = ["HONO", "NO", "NO2"]

def extract_and_filter_emissions(path, rows=ROWS, cols=COLS, vars_to_keep=TARGET_VARS, squeeze_lay=True, out=None):
    """Processes an emissions file for Texas and keeps only specified variables."""
    with xr.open_dataset(path) as ds:

        # 1. Decode IOAPI TFLAG into true hourly datetimes
        if "TFLAG" in ds.variables:
            tf = ds["TFLAG"].isel(VAR=0).values
            hh = tf[:, 1] // 10000  # Extract hours
            t = pd.to_datetime(tf[:, 0].astype(str), format="%Y%j") + pd.to_timedelta(hh, unit="h")

            sub = ds.assign_coords(TSTEP=t)
            sub = sub.drop_vars("TFLAG")
        else:
            sub = ds

        # 2. Slice for Texas window (ROWS and COLS)
        if rows is not None or cols is not None:
            sub = sub.isel(ROW=rows or slice(None), COL=cols or slice(None))

        # 3. Squeeze single layer if NLAYS = 1
        if "LAY" in sub.dims and sub.sizes["LAY"] == 1:
            sub = sub.squeeze("LAY", drop=True)

        # 4. Keep ONLY your target variables (+ coordinates like TSTEP, ROW, COL)
        existing_vars = [v for v in vars_to_keep if v in sub.variables]
        sub = sub[existing_vars]

        # 5. Load into memory before file closes
        sub = sub.load()

    if out:
        sub.to_netcdf(out)
        print(f"Successfully saved filtered file: {out}")

    return sub

In [ ]:
file_url = "/content/drive/MyDrive/Copy of model_ready_emis_2016_merged_nobeis_norwc_01_EQUATES_v1.0.tar"
# output_tar = "my_archive.tar"

# Download the file
# gdown.download(file_url, output_tar, quiet=False, fuzzy=True)

# 2. Inspect the contents without extracting
print("--- Files inside the archive ---")
with tarfile.open(file_url, 'r') as tar:
    # List all file names inside the tar archive
    for member_name in tar.getnames()[:50]:  # Limits preview to the first 50 files so it doesn't spam your screen
        print(member_name)

    total_files = len(tar.getnames())
    print(f"\nTotal files in archive: {total_files}")

In [ ]:
with tarfile.open(file_url, 'r') as tar:
    tar.extract('2016/merged_nobeis_norwc/emis_mole_all_20160111_12US1_nobeis_norwc_WR413_MYR_2016.nc4')
    tar.extract('2016/merged_nobeis_norwc/emis_mole_all_20160112_12US1_nobeis_norwc_WR413_MYR_2016.nc4')
    tar.extract('2016/merged_nobeis_norwc/emis_mole_all_20160113_12US1_nobeis_norwc_WR413_MYR_2016.nc4')
    tar.extract('2016/merged_nobeis_norwc/emis_mole_all_20160114_12US1_nobeis_norwc_WR413_MYR_2016.nc4')
    tar.extract('2016/merged_nobeis_norwc/emis_mole_all_20160115_12US1_nobeis_norwc_WR413_MYR_2016.nc4')
print("Extracted files successfully!")

In [ ]:
filepath=['/content/2016/merged_nobeis_norwc/emis_mole_all_20160111_12US1_nobeis_norwc_WR413_MYR_2016.nc4',
          '/content/2016/merged_nobeis_norwc/emis_mole_all_20160112_12US1_nobeis_norwc_WR413_MYR_2016.nc4',
          '/content/2016/merged_nobeis_norwc/emis_mole_all_20160113_12US1_nobeis_norwc_WR413_MYR_2016.nc4',
          '/content/2016/merged_nobeis_norwc/emis_mole_all_20160114_12US1_nobeis_norwc_WR413_MYR_2016.nc4',
          '/content/2016/merged_nobeis_norwc/emis_mole_all_20160115_12US1_nobeis_norwc_WR413_MYR_2016.nc4',
          ]

In [ ]:
for idx, name in enumerate(filepath, start=1):
    output_filename = f"emis_texas2016_jan{idx:02d}.nc4"
    emis  = extract_and_filter_emissions(path=name, rows=ROWS, cols=COLS,squeeze_lay=True, out=output_filename)

In [ ]:
ds = xr.open_dataset("/content/emis_texas2016_jan01.nc4")
# Access the time coordinate (replace 'time' with your actual variable name if different)
time_coord = ds["TSTEP"]
print(time_coord)

In [ ]:
# 1. Find all your saved daily Texas emission files (ensuring chronological order)
file_pattern = "met_texas016jan*.nc4"
files = sorted(glob.glob(file_pattern))

print(f"Found {len(files)} files to combine:")
for f in files:
    print(f" - {f}")

# 2. Define a preprocess function to slice each file BEFORE combining
def clean_daily_file(ds):
  # Keeps only the first 24 time steps (hours 00:00 to 23:00)
  # Drops the 25th step (midnight rollover into the next day)
  return ds.isel(TSTEP=slice(0, 24))

# 3. Open and concatenate all files along the time (TSTEP) dimension
with xr.open_mfdataset(
    files,
    combine="nested",
    concat_dim="TSTEP",
    preprocess=clean_daily_file,
    data_vars="minimal",
    coords="minimal",
    compat="override") as combined_ds:

    # 3. Save the merged dataset to a single file
    output_merged_file = "met_texas2016_jan01_15_combined.nc4"
    combined_ds.to_netcdf(output_merged_file)
    print(f"\nSuccessfully merged {len(files)} files into: {output_merged_file}")

    # Optional: print summary of the combined dataset
    print(combined_ds)

In [ ]:
ds_hourly = xr.open_dataset("met_texas2016_jan01_15_combined.nc4")

# 2. Resample to daily scale
# Use .mean() for daily average emission rates, or .sum() if you want cumulative totals
ds_daily = ds_hourly.resample(TSTEP="1D").mean()
ds_hourly = ds_hourly.drop_duplicates(dim="TSTEP")
# 3. Save the new daily dataset
output_daily_file = "met_texas2016_jan01_15_daily.nc4"
ds_daily.to_netcdf(output_daily_file)

print(f"Successfully converted to daily scale and saved to: {output_daily_file}")
print(ds_daily)

Successfully converted to daily scale and saved to: met_texas2016_jan01_15_daily.nc4
<xarray.Dataset> Size: 20MB
Dimensions:    (TSTEP: 15, ROW: 80, COL: 100)
Coordinates:
  * TSTEP      (TSTEP) datetime64[ns] 120B 2016-01-01 2016-01-02 ... 2016-01-15
Dimensions without coordinates: ROW, COL
Data variables: (12/41)
    PRSFC      (TSTEP, ROW, COL) float32 480kB 1.014e+05 1.013e+05 ... 9.685e+04
    USTAR      (TSTEP, ROW, COL) float32 480kB 0.2101 0.194 ... 0.4349 0.4412
    WSTAR      (TSTEP, ROW, COL) float32 480kB 0.6283 0.5668 ... 0.412 0.4191
    PBL        (TSTEP, ROW, COL) float32 480kB 300.3 248.1 187.9 ... 363.7 385.0
    ZRUF       (TSTEP, ROW, COL) float32 480kB 0.0002451 0.0002242 ... 0.1326
    MOLI       (TSTEP, ROW, COL) float32 480kB -0.04154 -0.04854 ... 0.004285
    ...         ...
    WSAT_PX    (TSTEP, ROW, COL) float32 480kB 1.0 1.0 0.4435 ... 0.4464 0.445
    WFC_PX     (TSTEP, ROW, COL) float32 480kB 1.0 1.0 0.2557 ... 0.2059 0.2273
    WWLT_PX    (TSTEP, ROW, CO

In [ ]:
ds=xr.open_dataset('/content/label_texas2016_jan01_15_daily.nc4')
ds

<xarray.Dataset> Size: 7MB
Dimensions:       (TSTEP: 15, ROW: 80, COL: 100)
Coordinates:
  * TSTEP         (TSTEP) datetime64[ns] 120B 2016-01-01 ... 2016-01-15
Dimensions without coordinates: ROW, COL
Data variables: (12/14)
    O3_MDA8       (TSTEP, ROW, COL) float32 480kB ...
    O3_AVG        (TSTEP, ROW, COL) float32 480kB ...
    CO_AVG        (TSTEP, ROW, COL) float32 480kB ...
    NO_AVG        (TSTEP, ROW, COL) float32 480kB ...
    NO2_AVG       (TSTEP, ROW, COL) float32 480kB ...
    SO2_AVG       (TSTEP, ROW, COL) float32 480kB ...
    ...            ...
    PM25_AVG      (TSTEP, ROW, COL) float32 480kB ...
    PM25_SO4_AVG  (TSTEP, ROW, COL) float32 480kB ...
    PM25_NO3_AVG  (TSTEP, ROW, COL) float32 480kB ...
    PM25_NH4_AVG  (TSTEP, ROW, COL) float32 480kB ...
    PM25_OC_AVG   (TSTEP, ROW, COL) float32 480kB ...
    PM25_EC_AVG   (TSTEP, ROW, COL) float32 480kB ...
Attributes: (12/33)
    IOAPI_VERSION:  $Id: @(#) ioapi library version 3.1 $                    ...
    EXEC_ID:        ????????????????                                         ...
    FTYPE:          1
    CDATE:          2021211
    CTIME:          215738
    WDATE:          2021211
    ...             ...
    VGLVLS:         [1.     0.9975]
    GDNAM:          12US1           
    UPNAM:          hr2day          
    VAR-LIST:       O3_MDA8         O3_AVG          CO_AVG          NO_AVG   ...
    FILEDESC:       Concentration file output                                ...
    HISTORY:

In [ ]:
ds=xr.open_dataset('/content/met_texas2016_jan01_15_daily.nc4')
ds

<xarray.Dataset> Size: 20MB
Dimensions:    (TSTEP: 15, ROW: 80, COL: 100)
Coordinates:
  * TSTEP      (TSTEP) datetime64[ns] 120B 2016-01-01 2016-01-02 ... 2016-01-15
Dimensions without coordinates: ROW, COL
Data variables: (12/41)
    PRSFC      (TSTEP, ROW, COL) float32 480kB ...
    USTAR      (TSTEP, ROW, COL) float32 480kB ...
    WSTAR      (TSTEP, ROW, COL) float32 480kB ...
    PBL        (TSTEP, ROW, COL) float32 480kB ...
    ZRUF       (TSTEP, ROW, COL) float32 480kB ...
    MOLI       (TSTEP, ROW, COL) float32 480kB ...
    ...         ...
    WSAT_PX    (TSTEP, ROW, COL) float32 480kB ...
    WFC_PX     (TSTEP, ROW, COL) float32 480kB ...
    WWLT_PX    (TSTEP, ROW, COL) float32 480kB ...
    CSAND_PX   (TSTEP, ROW, COL) float32 480kB ...
    FMSAND_PX  (TSTEP, ROW, COL) float32 480kB ...
    CLAY_PX    (TSTEP, ROW, COL) float32 480kB ...
Attributes: (12/33)
    IOAPI_VERSION:  ioapi-3.2: $Id: init3.F90 1 2017-06-10 18:05:20Z coats $ ...
    EXEC_ID:        mcip                                                     ...
    FTYPE:          1
    CDATE:          2020113
    CTIME:          14259
    WDATE:          2020113
    ...             ...
    VGLVLS:         [1.     0.9975]
    GDNAM:          12US1_CROSS     
    UPNAM:          OUTCM3IO        
    VAR-LIST:       PRSFC           USTAR           WSTAR           PBL      ...
    FILEDESC:       US EPA COMMUNITY MULTISCALE AIR QUALITY MODEL            ...
    HISTORY:

In [ ]:
ds=xr.open_dataset('/content/LULC_texas.nc4')
ds

<xarray.Dataset> Size: 224kB
Dimensions:  (LAY: 1, ROW: 80, COL: 100)
Dimensions without coordinates: LAY, ROW, COL
Data variables:
    LAT      (LAY, ROW, COL) float32 32kB ...
    LON      (LAY, ROW, COL) float32 32kB ...
    MSFX2    (LAY, ROW, COL) float32 32kB ...
    HT       (LAY, ROW, COL) float32 32kB ...
    DLUSE    (LAY, ROW, COL) float32 32kB ...
    LWMASK   (LAY, ROW, COL) float32 32kB ...
    PURB     (LAY, ROW, COL) float32 32kB ...
Attributes: (12/33)
    IOAPI_VERSION:  ioapi-3.2: $Id: init3.F90 1 2017-06-10 18:05:20Z coats $ ...
    EXEC_ID:        mcip                                                     ...
    FTYPE:          1
    CDATE:          2020113
    CTIME:          14259
    WDATE:          2020113
    ...             ...
    VGLVLS:         [1.     0.9975]
    GDNAM:          12US1_CROSS     
    UPNAM:          OUTGM3IO        
    VAR-LIST:       LAT             LON             MSFX2           HT       ...
    FILEDESC:       US EPA COMMUNITY MULTISCALE AIR QUALITY MODEL            ...
    HISTORY:

In [ ]:
ds=xr.open_dataset('/content/emis_texas2016_jan01.nc4')
ds

<xarray.Dataset> Size: 2MB
Dimensions:  (TSTEP: 25, ROW: 80, COL: 100)
Coordinates:
  * TSTEP    (TSTEP) datetime64[ns] 200B 2016-01-01 ... 2016-01-02
Dimensions without coordinates: ROW, COL
Data variables:
    HONO     (TSTEP, ROW, COL) float32 800kB ...
    NO       (TSTEP, ROW, COL) float32 800kB ...
    NO2      (TSTEP, ROW, COL) float32 800kB ...
Attributes: (12/34)
    IOAPI_VERSION:  ioapi-3.2: $Id: init3.F90 150 2020-04-11 17:51:44Z coats ...
    EXEC_ID:        ????????????????                                         ...
    FTYPE:          1
    CDATE:          2021007
    CTIME:          170954
    WDATE:          2021007
    ...             ...
    GDNAM:          12US1_459X299   
    UPNAM:          MRGGRID         
    VAR-LIST:       ACET            ACROLEIN        ALD2            ALD2_PRIM...
    FILEDESC:       Merged emissions output file from Mrggrid                ...
    HISTORY:        
    DESCRIPTION:    Sum of files:  /work/MOD3EVAL/callen05/WR413/WR413_MYR_20...

Convert to zarr

In [ ]:
filename = ['emis_texas2016_jan01_15_daily.nc4',
            'met_texas2016_jan01_15_daily.nc4',
            'LULC_texas.nc4',
            'label_texas2016_jan01_15_daily.nc4']

for name in filename:
  ds = xr.open_dataset(name)
  zarr_output_path = name.replace('.nc4', '.zarr')
  ds.to_zarr(zarr_output_path, mode='w')
  print(f'Successfully converted {name} -> {zarr_output_path}')

/usr/local/lib/python3.13/dist-packages/zarr/api/asynchronous.py:260: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Successfully converted emis_texas2016_jan01_15_daily.nc4 -> emis_texas2016_jan01_15_daily.zarr


/usr/local/lib/python3.13/dist-packages/zarr/api/asynchronous.py:260: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/zarr/api/asynchronous.py:260: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Successfully converted met_texas2016_jan01_15_daily.nc4 -> met_texas2016_jan01_15_daily.zarr
Successfully converted LULC_texas.nc4 -> LULC_texas.zarr
Successfully converted label_texas2016_jan01_15_daily.nc4 -> label_texas2016_jan01_15_daily.zarr


/usr/local/lib/python3.13/dist-packages/zarr/api/asynchronous.py:260: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


remove features from input files

In [ ]:
zarr_path = "/content/met_texas2016_jan01_15_daily.zarr"
ds = xr.open_zarr(zarr_path)
vars_to_keep = [
    "PRSFC",
    "TEMP2",
    "WSPD10",
    "WDIR10",
    "RGRND",
    "PBL",
    "Q2",
    "RN",
    "RC",
]

valid_vars = [v for v in vars_to_keep if v in ds.data_vars]
ds_cleaned = ds[valid_vars]
cleaned_zarr_path = "met_texas2016_jan01_15_cleaned.zarr"
ds_cleaned.to_zarr(cleaned_zarr_path, mode="w")

print(f"Original variable count: {len(ds.data_vars)}")
print(f"Kept {len(valid_vars)} essential variables.")
print(f"Cleaned dataset saved to '{cleaned_zarr_path}'.")

/usr/local/lib/python3.13/dist-packages/zarr/api/asynchronous.py:260: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Original variable count: 41
Kept 9 essential variables.
Cleaned dataset saved to 'met_texas2016_jan01_15_cleaned.zarr'.


In [ ]:
zarr_path = "/content/LULC_texas.zarr"
ds = xr.open_zarr(zarr_path)
vars_to_keep = [
    "LAT",
    "LON",
    "HT",
    "DLUSE",
    "LWMASK",
    "PURB",
]

valid_vars = [v for v in vars_to_keep if v in ds.data_vars]
ds_cleaned = ds[valid_vars]
cleaned_zarr_path = "LULC_texas_cleaned.zarr"
ds_cleaned.to_zarr(cleaned_zarr_path, mode="w")

print(f"Original variable count: {len(ds.data_vars)}")
print(f"Kept {len(valid_vars)} essential variables.")
print(f"Cleaned dataset saved to '{cleaned_zarr_path}'.")

Original variable count: 7
Kept 6 essential variables.
Cleaned dataset saved to 'LULC_texas_cleaned.zarr'.


/usr/local/lib/python3.13/dist-packages/zarr/api/asynchronous.py:260: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [ ]:
zarr_path = "/content/label_texas2016_jan01_15_daily.zarr"
ds = xr.open_zarr(zarr_path)
vars_to_keep = [
    "NO2_AVG",
]

valid_vars = [v for v in vars_to_keep if v in ds.data_vars]
ds_cleaned = ds[valid_vars]
cleaned_zarr_path = "label_texas2016_jan01_15_cleaned.zarr"
ds_cleaned.to_zarr(cleaned_zarr_path, mode="w")

print(f"Original variable count: {len(ds.data_vars)}")
print(f"Kept {len(valid_vars)} essential variables.")
print(f"Cleaned dataset saved to '{cleaned_zarr_path}'.")

Original variable count: 14
Kept 1 essential variables.
Cleaned dataset saved to 'label_texas2016_jan01_15_cleaned.zarr'.


/usr/local/lib/python3.13/dist-packages/zarr/api/asynchronous.py:260: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
